# RES-142 ? B2 Qwen3 scoring (Colab 2026.04 / A100)

**Offline A100 variant. All code cells remain unexecuted. The code cell opens a browser picker for exactly the eight hash-pinned files. No Colab CLI is used.**

This run uses explicit manual-operator shutdown authorization. `SESSION_WATCHDOG_QUALIFIED` remains `False`; the notebook does not claim an automatic shutdown controller. Stay at the computer for the entire run, start an external 25-minute timer when the A100 runtime connects, begin cleanup at 23 minutes, and manually disconnect/delete the runtime by 25 minutes.

The A100 variant changes only accelerator checks and operational gates. It preserves the G4 scientific scorer, Qwen3 model/tokenizer revisions, BF16, scoring template, frozen candidates, and RRF baseline.


## Before any paid connection

This notebook targets one NVIDIA A100 80GB with compute capability 8.0 and at least 75,000 MiB visible memory. It pins Colab runtime 2026.04, Linux Python 3.12, CUDA 12.9/cu129 packages, PyTorch, vLLM, BF16 scoring, model, and template.

**Manual operator mode:** remain at the computer and manually disconnect/delete the runtime. The notebook cannot terminate a hung browser or disconnect the runtime. Start an external 25-minute timer at GPU connection. Stop scoring and begin cleanup at 23 minutes; disconnect/delete by 25 minutes. The CU projection is not an enforced cap.

Execution remains blocked until fresh numeric spending approval, setup/rate/Drive authorizations, runtime confirmation, and the exact `APPROVE MANUAL SESSION SHUTDOWN` acknowledgment are entered. `SESSION_WATCHDOG_QUALIFIED` must remain `False`. Do not connect before those approvals.

The previous attempt's Drive evidence is immutable. This run uses the create-only directory `b2-attempt-20261021-a100-repair-94e06560a2534a2d99174badbbcf4062/`; the orchestrator refuses to reuse it.


In [ ]:
from __future__ import annotations

import hashlib
import json
import time
import traceback
from pathlib import Path

from google.colab import files

RUN_STARTED = time.monotonic()
SESSION_LIMIT_SECONDS = 25 * 60
WORK_LIMIT_SECONDS = 23 * 60

# Keep authorization fields closed until fresh numeric A100 spending approval.
RUNTIME_VERSION_CONFIRMED = ""
SETUP_AUTHORIZATION = ""
SHUTDOWN_MODE = "manual_operator"
SHUTDOWN_AUTHORIZATION = ""
RATE_INSPECTION_AUTHORIZATION = ""
RATE_INSPECTION_GPU = "A100"
RATE_INSPECTION_MAX_CU = 0.0
OBSERVED_CU_PER_HOUR = 0.0
APPROVED_MAX_CU = 0.0
DRIVE_AUTHORIZATION = ""
DRIVE_DESTINATION_CONFIRMATION = ""
SESSION_WATCHDOG_QUALIFIED = False

EXPECTED_SHA256 = {
    "res142_b2_colab.py": "f1554b33c725da3a0232b9c9e7b1f9f95cd5596862dc638a88169da83f67fc94",
    "res142_b2_runner.py": "7beb9fbf3a89deb8c47b32d39d85e1cc6636863256501a689aa39aca173f252b",
    "res142_b1_support.py": "74178e28b9cb8ac6e1a893e3fa07607a80869564caa6191ae81fbe8dde81f99e",
    "b2_execution.py": "2dc3ddd0a5a8aa9044d4eea38106e3920b36f8b741caca00d110ae4383b2701d",
    "res142-colab.lock.txt": "8b360f4533eb616eac38469e450d3df1a516f155afba464929036eada7d80e16",
    "res142_qwen3_reranker.jinja": "bf7b104801a4e4ad6a147d732e2d5a837a36f708cdbd1b871b864979d3e10acd",
    "res142-b2-candidate-input-v3.zip": "c98d26a7aaf38f35b66203f0d27a73f618800218b1a90c4e322a02ab2c4100ac",
    "rights-confirmation-v3.json": "65980a1e19e6a01444193df55e98fd90a941b4c14b4c83efbd32db3564449d92",
}
REQUIRED_FILES = set(EXPECTED_SHA256)
DRIVE_ROOT = "/content/drive/MyDrive/DynamisRAG/RES-142"


def require_positive_number(value: object, label: str) -> float:
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
        raise PermissionError(f"{label} must be entered as a positive number.")
    return float(value)


try:
    if SHUTDOWN_MODE != "manual_operator":
        raise PermissionError("This A100 package requires explicit manual operator shutdown mode.")
    if SHUTDOWN_AUTHORIZATION != "APPROVE MANUAL SESSION SHUTDOWN":
        raise PermissionError("Explicit manual session shutdown authorization is required.")
    if SESSION_WATCHDOG_QUALIFIED is not False:
        raise PermissionError("Manual mode must leave SESSION_WATCHDOG_QUALIFIED false.")
    if RUNTIME_VERSION_CONFIRMED != "2026.04":
        raise PermissionError("In Colab, select runtime 2026.04 and confirm it here.")
    if SETUP_AUTHORIZATION != "APPROVE COLAB SETUP":
        raise PermissionError("A fresh APPROVE COLAB SETUP authorization is required.")
    if RATE_INSPECTION_AUTHORIZATION != "APPROVE COLAB RATE INSPECTION":
        raise PermissionError("A fresh A100 rate inspection authorization is required.")
    if RATE_INSPECTION_GPU != "A100":
        raise PermissionError("The rate inspection and scoring target must both be A100.")
    if DRIVE_AUTHORIZATION != "APPROVE COLAB DRIVE STORAGE":
        raise PermissionError("A fresh Google Drive storage authorization is required.")
    if DRIVE_DESTINATION_CONFIRMATION.rstrip("/") != DRIVE_ROOT:
        raise PermissionError(f"Confirm the Drive root exactly as {DRIVE_ROOT!r}.")
    rate_limit = require_positive_number(RATE_INSPECTION_MAX_CU, "Rate-inspection maximum CU")
    observed_rate = require_positive_number(OBSERVED_CU_PER_HOUR, "Observed CU/hour rate")
    approved_max = require_positive_number(APPROVED_MAX_CU, "Approved maximum CU")
    projected_cu = observed_rate * SESSION_LIMIT_SECONDS / 3600
    if projected_cu > approved_max:
        raise PermissionError("The 25-minute CU projection exceeds a fresh numeric approval.")
    print("Start the external 25-minute timer at GPU connection; begin cleanup at 23 minutes.")
    print(json.dumps({
        "gpu": "A100",
        "session_limit_seconds": SESSION_LIMIT_SECONDS,
        "work_deadline_seconds": WORK_LIMIT_SECONDS,
        "observed_cu_per_hour": observed_rate,
        "projected_cu": projected_cu,
        "approved_max_cu": approved_max,
        "projection_is_platform_enforced": False,
    }, indent=2))

    uploaded = files.upload()
    if set(uploaded) != REQUIRED_FILES:
        raise ValueError(f"Select exactly the reviewed eight files: {sorted(REQUIRED_FILES)}.")
    input_dir = Path("/content/res142-b2-input")
    input_dir.mkdir(parents=True, exist_ok=False)
    for name in sorted(REQUIRED_FILES):
        data = uploaded[name]
        if not isinstance(data, bytes):
            raise TypeError(f"Uploaded artifact must be raw bytes: {name}")
        observed = hashlib.sha256(data).hexdigest()
        if observed != EXPECTED_SHA256[name]:
            raise ValueError(f"Uploaded artifact SHA-256 mismatch: {name}")
        path = input_dir / name
        with path.open("xb") as output_file:
            output_file.write(data)

    runner_path = input_dir / "res142_b2_colab.py"
    namespace = {"__name__": "res142_b2_colab_uploaded", "__file__": str(runner_path)}
    source = runner_path.read_text(encoding="utf-8")
    exec(compile(source, str(runner_path), "exec"), namespace)
    # Keep the watchdog signal false; pass manual authorization only after its explicit gate.
    namespace["WATCHDOG_QUALIFIED"] = SESSION_WATCHDOG_QUALIFIED
    namespace["MANUAL_OPERATOR_AUTHORIZED"] = True
    result = namespace["main"](
        upload_dir=input_dir,
        approvals={
            "setup": SETUP_AUTHORIZATION,
            "shutdown_mode": SHUTDOWN_MODE,
            "shutdown_authorization": SHUTDOWN_AUTHORIZATION,
            "rate_inspection": RATE_INSPECTION_AUTHORIZATION,
            "rate_inspection_gpu": RATE_INSPECTION_GPU,
            "rate_inspection_max_compute_units": rate_limit,
            "gpu": "A100",
            "observed_compute_units_per_hour": observed_rate,
            "approved_max_compute_units": approved_max,
            "drive": DRIVE_AUTHORIZATION,
            "drive_destination_confirmation": DRIVE_DESTINATION_CONFIRMATION,
            "colab_orchestrator_sha256": EXPECTED_SHA256["res142_b2_colab.py"],
        },
        run_started=RUN_STARTED,
        runtime_version_confirmed=RUNTIME_VERSION_CONFIRMED,
    )
    print(json.dumps(result, indent=2, sort_keys=True))
except BaseException as error:
    output_root = Path("/content/res142-b2-preflight")
    output_root.mkdir(parents=True, exist_ok=True)
    already_reported = any(Path("/content/res142-b2/output").glob("failure-*.json"))
    if not already_reported:
        failure_path = output_root / f"failure-{time.time_ns()}.json"
        failure_path.write_text(json.dumps({
            "stage": "notebook_preflight_or_execution",
            "error_type": type(error).__name__,
            "message": str(error)[:2000],
            "traceback": traceback.format_exc()[-12000:],
            "runtime_seconds": time.monotonic() - RUN_STARTED,
        }, indent=2, sort_keys=True), encoding="utf-8")
        try:
            files.download(str(failure_path))
        except Exception:
            pass
    print("Failure detected. Do not retry. Preserve the available evidence, then manually "
          "disconnect and delete the runtime now; no automatic shutdown is available.")
    raise


## Frozen scoring profile and stop controls

The reviewed B2 scorer remains pinned to Qwen/Qwen3-Reranker-0.6B revision and tokenizer revision `e61197ed45024b0ed8a2d74b80b4d909f1255473`, the exact Jinja template SHA-256, BF16, vLLM 0.31.0/cu129, and batches of 8. The A100 checks require compute capability 8.0, at least 75,000 MiB visible VRAM, a compatible CUDA 12.9 driver, PyTorch 2.13.0, and BF16 support.

The scoring algorithm, query/candidate order, candidate identities, and original RRF baseline are unchanged. A different GPU can produce small floating point score differences, so bitwise score identity is not claimed. Candidate inputs and ranking rules remain fixed.

The 23-minute work deadline is an in-runtime fail-safe. At that deadline the notebook stops scoring and begins cleanup. It reserves the remaining time for process termination and Drive flush; the operator must disconnect/delete the runtime no later than 25 minutes after GPU connection.


## Output and operator cleanup

The scorer persists each completed query and final evidence under the unique create-only attempt directory, with create-only writes and read-back hashes. It does not write to the previous attempt directory.

On success, check the final Drive flush status and manually disconnect/delete the runtime immediately. On failure, do not retry: preserve or download the failure evidence, and manually disconnect/delete the runtime immediately. Begin cleanup at 23 minutes and disconnect/delete by the hard 25-minute limit.
